In [ ]:
import os
import tensorflow as tf
import keras
import psutil
import tensorflow.keras.backend as K

print("TF version:", tf.__version__)
print("Keras version:", keras.__version__)
print("XLA Flags:", os.environ.get("TF_XLA_FLAGS"))
print("TF CUDA Built With:", tf.sysconfig.get_build_info()["cuda_version"])
print("GPU Available:", tf.config.list_physical_devices("GPU"))
print("TF_DISABLE_CUDNN_AUTOTUNE:", os.environ.get("TF_DISABLE_CUDNN_AUTOTUNE"))
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # '2' = only errors
K.clear_session()
print(f"RAM Available: {psutil.virtual_memory().available / (1024 ** 2):.2f} MB")


In [ ]:
!nvidia-smi

In [ ]:
import os
import zipfile
from multiprocessing import Pool, cpu_count
from pathlib import Path
from tqdm import tqdm

# ================================
# CONFIG
# ================================

SCRATCH_DATASET_DIR = Path.home() / "scratch" / "datasets"

ZIP_FILES = {
    SCRATCH_DATASET_DIR / "images_train.zip": SCRATCH_DATASET_DIR / "images_train",
    SCRATCH_DATASET_DIR / "images_test.zip": SCRATCH_DATASET_DIR / "images_test"
}

CHUNK_SIZE = 1000   # Files per process
NUM_WORKERS = min(cpu_count(), 30)

# ================================
# CLEAN & PREP FOLDERS
# ================================

for zip_path, out_folder in ZIP_FILES.items():
    if out_folder.exists():
        print(f"🧹 Removing existing folder: {out_folder}")
        os.system(f"rm -rf {out_folder}")
    os.makedirs(out_folder, exist_ok=True)
    print(f"Created folder: {out_folder}")

# ================================
# PARALLEL UNZIPPER
# ================================

def unzip_chunk(zip_file_path, output_dir, filenames):
    with zipfile.ZipFile(zip_file_path, 'r') as zf:
        for fname in filenames:
            try:
                zf.extract(fname, path=output_dir)
            except Exception as e:
                print(f"Error extracting {fname}: {e}")

def unzip_parallel(zip_file_path, output_dir):
    with zipfile.ZipFile(zip_file_path, 'r') as zf:
        all_files = zf.namelist()

    chunks = [all_files[i:i + CHUNK_SIZE] for i in range(0, len(all_files), CHUNK_SIZE)]
    args = [(zip_file_path, output_dir, chunk) for chunk in chunks]

    with Pool(NUM_WORKERS) as pool:
        list(tqdm(pool.starmap(unzip_chunk, args), total=len(chunks),
                  desc=f"Extracting {zip_file_path.name}"))

# ================================
# RUN UNZIPPING
# ================================

if __name__ == "__main__":
    for zip_file, output_folder in ZIP_FILES.items():
        unzip_parallel(zip_file, output_folder)

    print("\nAll zip files extracted successfully!")


In [ ]:
import os
import numpy as np
import random
import multiprocessing
from pathlib import Path
from tqdm import tqdm  # For progress bars

# Base path where datasets are stored
SCRATCH_PATH = Path.home() / "scratch" / "datasets"

# Define dataset directories (only Train and Test)
directories = {
    "Train Set": SCRATCH_PATH / "images_train",
    "Test Set": SCRATCH_PATH / "images_test"
}

BATCH_SIZE = 500  # Files per batch
NUM_WORKERS = min(multiprocessing.cpu_count(), 30)

def check_spectrogram(file_path):
    try:
        spectrogram = np.load(file_path, mmap_mode='r')
        file_name = os.path.basename(file_path)
        shape = spectrogram.shape
        is_zero = np.all(spectrogram == 0)
        has_nan = np.isnan(spectrogram).any()
        has_inf = np.isinf(spectrogram).any()
        return file_name, shape, is_zero, has_nan, has_inf, None
    except Exception as e:
        return file_path, None, None, None, None, str(e)

def process_spectrograms_in_batches(file_list, dataset_name):
    shape_set = set()
    corrupt_files = []
    zero_files = []
    nan_files = []
    inf_files = []
    chunks = [file_list[i:i + BATCH_SIZE] for i in range(0, len(file_list), BATCH_SIZE)]
    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        for chunk in tqdm(chunks, desc=f"🔍 Checking {dataset_name} in batches"):
            results = pool.map(check_spectrogram, chunk)
            for file_name, shape, is_zero, has_nan, has_inf, error in results:
                if error:
                    corrupt_files.append(file_name)
                else:
                    shape_set.add(shape)
                    if is_zero:
                        zero_files.append(file_name)
                    if has_nan:
                        nan_files.append(file_name)
                    if has_inf:
                        inf_files.append(file_name)
    return shape_set, corrupt_files, zero_files, nan_files, inf_files

def check_spectrograms_parallel(directory, dataset_name):
    absolute_dir = Path(directory).resolve()
    print(f"\nChecking dataset: {dataset_name}")
    print(f"Looking in folder: {absolute_dir}")
    if not absolute_dir.exists():
        print(f"Warning: Directory '{directory}' does not exist!")
        return
    npy_files = list(absolute_dir.glob("*.npy"))
    print(f"Found {len(npy_files)} .npy files in {dataset_name}.")
    if npy_files:
        print(f"Sample files: {random.sample(npy_files, min(5, len(npy_files)))}")
    if not npy_files:
        print(f"No spectrogram files found in {dataset_name} ({directory})!")
        return
    shape_set, corrupt_files, zero_files, nan_files, inf_files = process_spectrograms_in_batches(npy_files, dataset_name)
    print(f"\n**Results for {dataset_name}** 📊")
    print(f"Checked {len(npy_files)} files.")
    print(f"Unique spectrogram shapes: {shape_set}")
    print(f"{len(corrupt_files)} corrupt files: {corrupt_files}")
    print(f"{len(zero_files)} all-zero files: {zero_files}")
    print(f"{len(nan_files)} files contain NaN: {nan_files}")
    print(f"{len(inf_files)} files contain Inf: {inf_files}")
    debug_files = random.sample(npy_files, min(5, len(npy_files)))
    print("\n🔍 **Sample Spectrograms for Debugging**:")
    for file_path in debug_files:
        spec_img = np.load(file_path, mmap_mode='r')
        print(f"{file_path.name} | Shape: {spec_img.shape} | Min: {np.min(spec_img)} | Max: {np.max(spec_img)} | Mean: {np.mean(spec_img)}")
    print("-" * 50)

if __name__ == "__main__":
    for dataset_name, directory in directories.items():
        check_spectrograms_parallel(directory, dataset_name)


In [ ]:
import os
import numpy as np
import multiprocessing
from pathlib import Path
from tqdm import tqdm

# Define dataset directories in scratch
SCRATCH_DATASET_PATH = Path.home() / "scratch" / "datasets"
train_dir = SCRATCH_DATASET_PATH / "images_train"
test_dir  = SCRATCH_DATASET_PATH / "images_test"

# Ensure directories exist
for directory in [train_dir, test_dir]:
    if not directory.exists():
        raise FileNotFoundError(f"Error: Directory '{directory}' does not exist!")

# Parallelism settings
CHUNK_SIZE = 500
NUM_WORKERS = min(multiprocessing.cpu_count(), 30)
THRESHOLD = 1e-3

# Get spectrogram filenames
train_files = {f.name for f in train_dir.iterdir() if f.suffix == ".npy" and "_aug" not in f.name}
test_files  = {f.name for f in test_dir.iterdir() if f.suffix == ".npy"}

# Find common filenames
common_train_test = list(train_files.intersection(test_files))

# Function to process a chunk of files
def process_chunk(file_chunk, dir1, dir2):
    exact_duplicates = 0
    near_duplicates = 0
    for filename in file_chunk:
        file_path_1 = dir1 / filename
        file_path_2 = dir2 / filename
        try:
            spec1 = np.load(file_path_1, mmap_mode='r')
            spec2 = np.load(file_path_2, mmap_mode='r')
            if np.array_equal(spec1, spec2):
                exact_duplicates += 1
            elif np.allclose(spec1, spec2, atol=THRESHOLD):
                near_duplicates += 1
        except Exception as e:
            print(f"Error processing {filename}: {e}")
    return exact_duplicates, near_duplicates

# 🔁 Wrapper
def process_chunk_wrapper(args):
    return process_chunk(*args)

# 🔄 Multiprocessing executor
def process_spectrograms_parallel(common_files, dir1, dir2, description):
    if not common_files:
        return 0, 0
    exact_total, near_total = 0, 0
    chunks = [common_files[i:i + CHUNK_SIZE] for i in range(0, len(common_files), CHUNK_SIZE)]
    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        results = list(tqdm(pool.imap(process_chunk_wrapper, [(chunk, dir1, dir2) for chunk in chunks]),
                            total=len(chunks), desc=f"Checking Duplicates ({description})"))
    for exact, near in results:
        exact_total += exact
        near_total += near
    return exact_total, near_total

# ✅ Run check
if __name__ == "__main__":
    exact_train_test, near_train_test = process_spectrograms_parallel(
        common_train_test, train_dir, test_dir, "Train vs Test"
    )

    # 📊 Final Report
    print("\n**Dataset Uniqueness Report (Train vs Test Only)**")
    print(f"Shared filenames: {len(common_train_test)}")

    print(f"\nExact duplicate spectrograms:")
    print(f"   Train vs Test: {exact_train_test}")

    print(f"\nNear-duplicate spectrograms:")
    print(f"   Train vs Test: {near_train_test}")

    if exact_train_test == 0 and near_train_test == 0:
        print("**Success! Train and Test datasets are completely distinct.**")
    else:
        print("**Warning:** Duplicate or near-duplicate files found between Train and Test.")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

# Define full dataset paths
SCRATCH_DATASET_PATH = Path.home() / "scratch" / "datasets"
directories = {
    "Train Set": SCRATCH_DATASET_PATH / "images_train",
    "Test Set": SCRATCH_DATASET_PATH / "images_test"
}

BATCH_SIZE = 1000  # Process files in chunks

def get_spectrogram_chunks(directory, batch_size=BATCH_SIZE):
    if not directory.exists():
        print(f"Warning: Directory '{directory}' not found!")
        return []
    spectrogram_files = list(directory.glob("*.npy"))
    if not spectrogram_files:
        print(f"No spectrograms found in {directory}!")
        return []
    random.shuffle(spectrogram_files)
    return [spectrogram_files[i : i + batch_size] for i in range(0, len(spectrogram_files), batch_size)]

def get_random_spectrogram(directory):
    spectrogram_chunks = get_spectrogram_chunks(directory)
    if not spectrogram_chunks:
        return None, None
    random_chunk = random.choice(spectrogram_chunks)
    random_file = random.choice(random_chunk)
    spectrogram = np.load(random_file)
    return spectrogram, random_file.name

train_spectrogram, train_filename = get_random_spectrogram(directories["Train Set"])
test_spectrogram, test_filename = get_random_spectrogram(directories["Test Set"])

if train_spectrogram is not None and test_spectrogram is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    im1 = axes[0].imshow(train_spectrogram, aspect='auto', cmap='inferno')
    axes[0].set_title(f"Train Set\n{train_filename}")
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel("Frequency")
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

    im2 = axes[1].imshow(test_spectrogram, aspect='auto', cmap='inferno')
    axes[1].set_title(f"Test Set\n{test_filename}")
    axes[1].set_xlabel("Time")
    axes[1].set_ylabel("Frequency")
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
else:
    print("Error: Could not find spectrograms in one or both datasets.")


In [ ]:
import os
import numpy as np
import re
import multiprocessing
import collections
from pathlib import Path
from tqdm import tqdm

# Constants
IMAGE_HEIGHT, IMAGE_WIDTH, N_CHANNELS = 96, 96, 1
N_CLASSES = 10
BATCH_SIZE = 500
NUM_WORKERS = min(30, multiprocessing.cpu_count())

# Define full paths to datasets in scratch
SCRATCH_DATA_PATH = Path.home() / "scratch" / "datasets"
SPECTROGRAM_DIRS = {
    "Train Set": SCRATCH_DATA_PATH / "images_train",
    "Test Set": SCRATCH_DATA_PATH / "images_test",
}

FILENAME_PATTERN = re.compile(r'(\d+)_(\d+)_(\d+)(?:_aug\d+)?\.npy')

def extract_label(filename):
    match = FILENAME_PATTERN.match(filename)
    if match:
        digit = int(match.group(1))
        speaker = int(match.group(2))
        return digit, speaker
    return None, None

def process_spectrogram(file_path):
    filename = os.path.basename(file_path)
    label, speaker = extract_label(filename)
    if label is None or not (0 <= label < N_CLASSES):
        return None
    try:
        spectrogram = np.load(file_path, mmap_mode='r')
        spectrogram = np.expand_dims(spectrogram, axis=-1)  # add channel dimension
        spectrogram = (spectrogram + 80) / 80  # normalize to [0, 1]
        return spectrogram, label, speaker, filename
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def process_batch(file_paths):
    return [res for res in map(process_spectrogram, file_paths) if res is not None]

def process_dataset(directory, dataset_name):
    print(f"\nProcessing dataset: {dataset_name}")
    if not directory.exists():
        print(f"Warning: Directory '{directory}' does not exist!")
        return np.array([]), np.array([]), np.array([]), []
    npy_files = list(directory.glob("*.npy"))
    if not npy_files:
        print(f"No spectrogram files found in {dataset_name} ({directory})!")
        return np.array([]), np.array([]), np.array([]), []
    chunks = [npy_files[i:i + BATCH_SIZE] for i in range(0, len(npy_files), BATCH_SIZE)]
    with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
        results = list(tqdm(pool.imap(process_batch, chunks), total=len(chunks), desc=f"🔍 {dataset_name}"))
    spectrograms, labels, speakers, filenames = [], [], [], []
    for batch in results:
        for spectrogram, label, speaker, filename in batch:
            spectrograms.append(spectrogram)
            labels.append(label)
            speakers.append(speaker)
            filenames.append(filename)
    return np.array(spectrograms, dtype=np.float32), np.array(labels, dtype=np.int32), np.array(speakers, dtype=np.int32), filenames

if __name__ == "__main__":
    dataset_results = {}
    for dataset_name, directory in SPECTROGRAM_DIRS.items():
        X, y, speaker_ids, file_names = process_dataset(directory, dataset_name)
        dataset_results[dataset_name] = (X, y, speaker_ids, file_names)
        if X.size == 0:
            raise ValueError(f"No spectrograms were loaded for {dataset_name}! Check file paths.")
        label_counts = collections.Counter(y)
        speaker_counts = collections.Counter(speaker_ids)
        print(f"\n**{dataset_name} Summary** ")
        print(f"Total Spectrograms: {X.shape[0]}")
        print(f"Unique Labels (Digits): {np.unique(y)}")
        print(f"Class Distribution: {label_counts}")
        print(f"Speaker Distribution: {speaker_counts}")
        print(f"Sample Filenames: {file_names[:5]}")
        print("-" * 50)


In [ ]:
# Cell 1
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint, Callback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
import math
import os
import random
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils.dataframe import dataframe_to_rows
import visualkeras
import re

# Constants
IMAGE_HEIGHT = 96
IMAGE_WIDTH = 96
N_CHANNELS = 1
N_CLASSES = 10
BATCH_SIZE = 32

# Load pre-processed normalized datasets
X_train, y_train, _, _ = dataset_results["Train Set"]
X_test, y_test, _, test_filenames = dataset_results["Test Set"]

print("Dataset loaded:", X_train.shape, y_train.shape)


In [ ]:
# Cell 2

def build_base_cnn():
    model = tf.keras.models.Sequential([
        layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, N_CHANNELS)),
        layers.Conv2D(32, 3, strides=2, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(N_CLASSES, activation='softmax')
    ])
    return model

def build_small_cnn():
    model = tf.keras.models.Sequential([
        layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, N_CHANNELS)),
        layers.Conv2D(16, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(N_CLASSES, activation='softmax')
    ])
    return model

def build_deep_spp_cnn():
    def conv_block(filters):
        return [
            layers.Conv2D(filters, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
            layers.BatchNormalization(),
            layers.ReLU()
        ]
    return tf.keras.Sequential([
        layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, N_CHANNELS)),
        *conv_block(64), *conv_block(64), layers.MaxPooling2D(2),
        *conv_block(128), *conv_block(128), layers.MaxPooling2D(2),
        *conv_block(256), *conv_block(256), layers.MaxPooling2D(2),
        *conv_block(512), *conv_block(512), layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(256, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(128, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.5),
        layers.Dense(N_CLASSES, activation='softmax', dtype=tf.float32)
    ])

def build_residual_face():
    def residual_block(x, out_channels):
        shortcut = x
        x = layers.DepthwiseConv2D(3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU()(x)
        x = layers.Conv2D(out_channels, 1, padding='same', use_bias=False,
                          kernel_regularizer=regularizers.l2(1e-3))(x)
        x = layers.BatchNormalization()(x)
        if shortcut.shape[-1] != out_channels:
            shortcut = layers.Conv2D(out_channels, 1, padding='same', use_bias=False)(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        x = layers.Add()([x, shortcut])
        return layers.LeakyReLU()(x)

    inputs = layers.Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, N_CHANNELS))
    x = layers.Conv2D(16, 3, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.MaxPooling2D(2)(x)
    for filters in [32, 64, 128, 128]:
        x = residual_block(x, filters)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(N_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)


In [ ]:
# cell 3
current_augment_prob = 0.0

def get_current_augment_prob():
    global current_augment_prob
    return current_augment_prob

def static_spec_augment(spectrogram, label):
    def augment(x):
        x = tf.image.random_flip_left_right(x)
        x = tf.image.random_brightness(x, 0.1)
        return x
    return augment(spectrogram), label

def dynamic_spec_augment():
    def augment(x, y):
        prob = get_current_augment_prob()
        x = tf.cond(tf.random.uniform([]) < prob, lambda: tf.image.random_flip_left_right(x), lambda: x)
        x = tf.cond(tf.random.uniform([]) < prob, lambda: tf.image.random_brightness(x, 0.1), lambda: x)
        return x, y
    return augment

class AugmentProbScheduler(Callback):
    def __init__(self, start_prob=0.0, end_prob=0.25, step_epochs=10):
        super().__init__()
        self.start_prob = start_prob
        self.end_prob = end_prob
        self.step_epochs = step_epochs
    def on_epoch_begin(self, epoch, logs=None):
        global current_augment_prob
        steps = epoch // self.step_epochs
        new_prob = self.start_prob + (0.05 * steps)
        current_augment_prob = min(new_prob, self.end_prob)
        print(f"Epoch {epoch+1}: Augment prob = {current_augment_prob:.2f}")

def prepare_dataset(X, y, batch_size, shuffle=True, augment_type='none'):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(len(X))
    if augment_type == 'static':
        ds = ds.map(static_spec_augment, num_parallel_calls=tf.data.AUTOTUNE)
    elif augment_type == 'dynamic':
        ds = ds.map(dynamic_spec_augment(), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [ ]:
# cell 4
def get_callbacks(mode):
    if mode == 'val_loss':
        return [ModelCheckpoint("best_model.keras", monitor="val_loss", save_best_only=True, verbose=0),
                EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=5e-5)]
    elif mode == 'val_accuracy':
        return [ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True, verbose=0),
                EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)]
    elif mode == 'simple_reduce':
        return [ModelCheckpoint("best_model.keras", monitor="val_loss", save_best_only=True, verbose=0),
                ReduceLROnPl ( ateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)]
    elif mode == 'long_dynamic':
        return [ModelCheckpoint("best_model.keras", monitor="val_loss", save_best_only=True, verbose=0),
                ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
                EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=0),
                AugmentProbScheduler()]
    

In [ ]:
# cell 5

# run this when you want to start from scratch
# results = []
import tensorflow.keras.backend as K

def sanitize_filename(name):
    return re.sub(r'[^\w\-_\. ]', '_', name)

def train_combination(model_fn, callbacks_mode, augment_mode, tag, epochs=20):
    # Clear any leftover model before building a new one
    K.clear_session()
    
    print(f"Training: {tag}")
    
    model = model_fn()
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    train_ds = prepare_dataset(X_train, y_train, BATCH_SIZE, augment_type=augment_mode)
    test_ds = prepare_dataset(X_test, y_test, BATCH_SIZE, shuffle=False)

    history = model.fit(train_ds, validation_data=test_ds,
                        epochs=epochs, callbacks=get_callbacks(callbacks_mode), verbose=1)

    loss, acc = model.evaluate(test_ds, verbose=0)
    predictions = model.predict(test_ds)
    probs = np.max(predictions, axis=1)
    preds = np.argmax(predictions, axis=1)

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(history.history['loss'], label='Train')
    ax[0].plot(history.history['val_loss'], label='Val')
    ax[0].set_title("Loss")
    ax[0].legend()
    ax[1].plot(history.history['accuracy'], label='Train')
    ax[1].plot(history.history['val_accuracy'], label='Val')
    ax[1].set_title("Accuracy")
    ax[1].legend()
    plt.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    plt.close()
    
    # Save architecture visualization to file
    arch_path = f"architecture_{sanitize_filename(tag)}.png"
    visualkeras.layered_view(model, to_file=arch_path, legend=True)


    results.append({
        'tag': tag,
        'acc': acc,
        'loss': loss,
        'history': history.history,
        'plot': buf,
        'probs': probs.tolist(),
        'preds': preds.tolist(),
        'y_true': y_test.tolist(),
        'epochs_run': len(history.history['loss']),  # optional
        'best_val_acc': max(history.history['val_accuracy']),
        'best_val_loss': min(history.history['val_loss']),
        'train_acc': history.history['accuracy'][-1],
        'train_loss': history.history['loss'][-1],
        'architecture_plot': arch_path
    })

    # Free up memory from the current model
    K.clear_session()
    

In [ ]:
# cell 6

models_dict = {
    "BaseCNN": build_base_cnn,
    "SmallCNN": build_small_cnn,
    "DeepSPP": build_deep_spp_cnn,
    "ResidualFace": build_residual_face
}

pipelines = {
    "A_val_loss": ("val_loss", 20),
    "B_val_acc": ("val_accuracy", 50),
    "C_reduce": ("simple_reduce", 20),
    "D_long_dynamic": ("long_dynamic", 100)
}

augment_types = {
    "None": "none",
    "Static": "static",
    "Dynamic": "dynamic"
}

In [ ]:
# ─── Run only DeepSPP | D_long_dynamic | None ─────────────────────

import tensorflow.keras.backend as K

# clear any previous state
K.clear_session()

# tag for this run
tag = "DeepSPP | D_long_dynamic | None"

# pull out the right callback‐mode and epoch count
cb_mode, epochs = pipelines["D_long_dynamic"]     # ("long_dynamic", 100)
aug_mode       = augment_types["None"]            # "none"

# run the one model/pipeline/augment combo
train_combination(
    models_dict["DeepSPP"],
    cb_mode,
    aug_mode,
    tag,
    epochs=epochs
)

print("Done. Your best‐version is saved in best_model.keras")
